In [1]:
!pip list

Package                   Version
------------------------- --------------
accelerate                1.10.1
adapters                  1.1.0
aiohappyeyeballs          2.6.1
aiohttp                   3.11.15
aiosignal                 1.3.2
amdsmi                    24.6.3+52b3947
annotated-types           0.7.0
ansible-core              2.14.18
anyio                     4.7.0
argon2-cffi               23.1.0
argon2-cffi-bindings      21.2.0
arrow                     1.3.0
asttokens                 3.0.0
async-lru                 2.0.4
async-timeout             5.0.1
attrs                     24.3.0
babel                     2.16.0
beautifulsoup4            4.12.3
bleach                    6.2.0
blinker                   1.9.0
Bottleneck                1.5.0
certifi                   2024.12.14
cffi                      1.17.1
chardet                   4.0.0
charset-normalizer        3.4.0
clang                     20.1.5
click                     8.1.8
cockpit                   334.1
col

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append("..")

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
from src.models.qwen3 import Qwen3

t = Qwen3(model_name = "Qwen/Qwen3-0.6B", device = "mps")

t.model

INFO:root:Using device: mps


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [13]:
from src.datasets.lener import LenerDataset

l = LenerDataset(tokenizer=t.tokenizer)

In [14]:
data = l.load_dataset()
data

INFO:src.datasets.lener:Loading dataset: peluz/lener_br
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'peluz/lener_br' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'peluz/lener_br' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Using the latest cached version of the dataset since peluz/lener_br couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'lener_br' at /Users/mstauffer/.cache/huggingface/datasets/peluz___lener_br/lener_br/1.0.0/4a8c97e6813b5c2d85a50faf0a3e6c24ea82f4a9044e6e9e8b24997d27399

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'prompt', 'ground_truth'],
        num_rows: 7828
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'prompt', 'ground_truth'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'prompt', 'ground_truth'],
        num_rows: 1390
    })
})

In [9]:
data["train"][1]["ground_truth"]

'LEGISLACAO: art . 178 , II , do CPC; ORGANIZACAO: Ministério Público; ORGANIZACAO: Ministério Público'

In [15]:
from src.peft_configs.lora import LoraAdapter
# PRESET = 'baseline'
PRESET = 'full_attention'
# PRESET = 'all'
# PRESET = 'ffn'
# PRESET = 'full_attention_plus_ffn'
# PRESET = 'low_rank'
# PRESET = 'high_rank'
# PRESET = 'high_rank_XX'

lora = LoraAdapter(
    model=t.model,
    lora_preset = PRESET
)

model = lora.apply_lora()
model

INFO:root:Applying LoRA with preset: full_attention
INFO:root:LoRA configurations: {'r': 32, 'lora_alpha': 64, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'], 'lora_dropout': 0.1, 'bias': 'none', 'task_type': <TaskType.CAUSAL_LM: 'CAUSAL_LM'>}


trainable params: 9,175,040 || all params: 605,224,960 || trainable%: 1.5160


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [25]:
import torch
from trl import GRPOTrainer, GRPOConfig
from src.models.qwen3 import Qwen3
from src.datasets.lener import LenerDataset
from src.peft_configs.lora import LoraAdapter
from src.rewards.ner_reward import ner_reward_func

# ---------------------------------------------------------------------------
# Configuration — adjust these paths/settings before running on the cluster
# ---------------------------------------------------------------------------
MODEL_NAME    = "Qwen/Qwen3-8B"
LORA_PRESET   = "high_rank_64"
CHECKPOINT_NAME = f"Qwen3-8B_{LORA_PRESET}_grpo"

# Path to a finished SFT checkpoint to warm-start GRPO from.
# GRPO needs the model to already know the output format, otherwise the reward
# signal is too noisy at the start of training.
# Set to None to train from the base model (not recommended).
SFT_ADAPTER_PATH = "./outputs/checkpoints/lener_br/Qwen3-8B_high_rank_64/checkpoint-2450"

# ---------------------------------------------------------------------------
# 1. Load model (base + SFT adapter warm-start)
# ---------------------------------------------------------------------------
t = Qwen3(model_name=MODEL_NAME, device="cuda:0")
t.model = t.load_model(adapter_path=SFT_ADAPTER_PATH)

# ---------------------------------------------------------------------------
# 2. Load dataset in GRPO format: {'prompt', 'ground_truth'}
# ---------------------------------------------------------------------------
l = LenerDataset(tokenizer=t.tokenizer)
data = l.load_dataset(format="grpo")

# ---------------------------------------------------------------------------
# 3. Apply a fresh LoRA adapter on top of the SFT-warmed model for GRPO
#    (the SFT adapter is already merged into the base via PeftModel; we add a
#    separate trainable LoRA for the GRPO update step)
# ---------------------------------------------------------------------------
lora = LoraAdapter(model=t.model, lora_preset=LORA_PRESET)
model = lora.apply_lora()

# ---------------------------------------------------------------------------
# 4. GRPO training
# ---------------------------------------------------------------------------
training_args = GRPOConfig(
    output_dir=f"./outputs/checkpoints/lener_br/{CHECKPOINT_NAME}",
    num_train_epochs=3,
    per_device_train_batch_size=2,       # GRPO memory = batch * num_generations
    gradient_accumulation_steps=8,       # effective batch = 2 * 8 = 16 prompts
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    learning_rate=5e-6,                  # lower than SFT; we refine, not re-learn
    fp16=False,
    bf16=True,                           # bfloat16 is stable on AMD ROCm
    report_to="tensorboard",
    logging_dir=f"./runs/grpo/{CHECKPOINT_NAME}",
    # GRPO-specific
    num_generations=8,                   # samples per prompt for advantage estimation
    max_prompt_length=1024,
    max_completion_length=256,           # entity-extraction answers are short
    beta=0.04,                           # KL penalty (low: model already knows format)
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=data["train"],
    processing_class=t.tokenizer,
    reward_funcs=ner_reward_func,
)

trainer.train()

/Users/mstauffer/Documents/unb/research/decoder-lener-iob/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


TrainOutput(global_step=2, training_loss=0.00010494186426512897, metrics={'train_runtime': 1973.9803, 'train_samples_per_second': 0.005, 'train_steps_per_second': 0.001, 'total_flos': 0.0, 'train_loss': 0.00010494186426512897})

In [27]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [ ]:
from src.scripts.utils import predict_entities_batch

texts = [
    # "O ministério público acatou a decisão do STF e pediu a suspensão do processo contra o ex-presidente Lula, com base na Lei 17.681/2017.",
    "- Tratando-se de ação indenizatória ajuizada por pessoa incapaz , é obrigatória a intervenção do Ministério Público na condição de fiscal da ordem jurídica ."
]

results = predict_entities_batch(texts, model, t.tokenizer)
for res in results:
    print(res)

In [28]:
results

['Você é um especialista jurídico responsável por identificar entidades em textos.        \nAs entidades que você deve identificar são:\n\n- ORGANIZAÇÃO: Refere-se a entidades que representam organizações, como empresas, instituições governamentais, ONGs, etc.\n- PESSOA: Designa entidades que são nomes de pessoas físicas.\n- TEMPO: Marca entidades que expressam informações temporais, como datas, horários, períodos, etc.\n- LOCAL: Indica entidades que representam lugares geográficos, como cidades, países, estados, endereços, etc.\n- LEGISLAÇÃO: Identifica entidades que correspondem a Atos de Lei, como leis, decretos, portarias, etc.\n- JURISPRUDÊNCIA: Assinala entidades que se referem a decisões relativas a casos legais.      \n\nsegue o texto\n- Tratando-se de ação indenizatória ajuizada por pessoa incapaz , é obrigatória a intervenção do Ministério Público na condição de fiscal da ordem jurídica .\nResposta:\n-\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\